In [54]:
%load_ext autoreload
%autoreload 2
from src.data_loader import load_and_clean_data
from IPython.display import display, HTML
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import textwrap
from src.analysis import *
import numpy as np

df = load_and_clean_data("../data/raw")

df.info()


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
<class 'pandas.core.frame.DataFrame'>
Index: 121573 entries, 0 to 121992
Data columns (total 17 columns):
 #   Column             Non-Null Count   Dtype              
---  ------             --------------   -----              
 0   timestamp          121573 non-null  datetime64[ns, UTC]
 1   duration           121573 non-null  object             
 2   song               121018 non-null  object             
 3   artist             121018 non-null  object             
 4   album              121018 non-null  object             
 5   country            121573 non-null  object             
 6   device             121573 non-null  object             
 7   ms_played          121573 non-null  int64              
 8   track_id           121018 non-null  object             
 9   reason_start       121573 non-null  object             
 10  reason_end         121573 non-null  object             
 11  shuffle 

In [ ]:

display(HTML("<style>.container { width:100% !important; }</style>"))

pd.set_option('display.expand_frame_repr', False)

In [ ]:
top_artist= get_top_ranking( df, "artist" , 10 )
display(top_artist)
top_song= get_top_ranking(df, "song" , 10 )

In [ ]:

fav_song_of_top_artist = (df.groupby(['artist', 'song'])['ms_played'].
                          sum().
                          reset_index().
                          sort_values(by='ms_played', ascending=False)
                          )
fav_song_of_top_artist= fav_song_of_top_artist.drop_duplicates(subset=['artist'], keep='first')
fav_song_of_top_artist['hours']= (fav_song_of_top_artist['ms_played'] / 3600000).round(1)
fav_song_of_top_artist= pd.merge(top_artist, fav_song_of_top_artist, on=['artist'] )

print(top_song.head(10))
print(top_artist.head(10))
print(fav_song_of_top_artist.head(10))


In [ ]:

plt.figure(figsize=(12, 6))
sns.set_style("whitegrid")


ax = sns.barplot(
    data=top_artist.head(10),
    x='artist',
    y='hours',
    hue='artist',
    palette='viridis',
    legend=False
)

plt.title('Top 10 Artyści', fontsize=16, fontweight='bold')
plt.xlabel('')
plt.ylabel('Godziny słuchania', fontsize=12)
plt.xticks(rotation=45, ha='right')


for container in ax.containers:
    ax.bar_label(container, fmt='%.0f h', padding=0)

plt.tight_layout()
plt.show()

In [ ]:
df_plot = top_song.head(10).copy()
plt.figure(figsize=(12,6))

ax = sns.barplot(
    data = df_plot,
    x='song',
    y='hours',
    hue='song',
    palette='seismic',
    legend=False
)
plt.title('Top 10 Song', fontsize=16, fontweight='bold')
plt.xlabel('')
plt.ylabel('czas słuchania [h]', fontsize=10)
plt.xticks(rotation=45, ha='right')

hours_labels = [f"{float(h)}h" for h in df_plot['hours']]
streams_labels = [f"{int(s)}x" for s in df_plot['streams']]

for i, container in enumerate(ax.containers):
    ax.bar_label(container, labels = [hours_labels[i]] , label_type= 'edge',padding=0)
    ax.bar_label(container, labels = [streams_labels[i]] ,label_type= 'center', padding=0)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
df_plot = top_song.head(15).sort_values('hours')


plt.hlines(y=df_plot['song'], xmin=0, xmax=df_plot['hours'], color='grey', alpha=0.4)

# 2. Rysujemy "lizaki" (kropki na końcach)
plt.scatter(df_plot['hours'], df_plot['song'], color='#1DB954', s=100, alpha=1)

# 3. Dodajemy tekst z liczbą odtworzeń obok kropki
for i, (h, s) in enumerate(zip(df_plot['hours'], df_plot['streams'])):
    plt.text(h + 0.5, i, f"{int(s)}x", va='center', fontsize=10, color='blue', fontweight='bold')

plt.title('Top Utwory: Godziny i Częstotliwość (x)', fontsize=16)
plt.xlabel('Suma godzin')
plt.ylabel('')
sns.despine(left=True, bottom=True) # Usuwamy zbędne ramki
plt.show()

In [ ]:

plt.figure(figsize=(12, 8))
df_plot = top_song.head(15) # Weźmy 15 piosenek dla lepszego efektu

# Tworzymy wykres punktowy
sns.scatterplot(
    data=df_plot,
    x='streams',
    y='hours',
    size='hours',        # Wielkość kropki zależy od czasu
    hue='song',          # Kolor dla każdej piosenki
    legend=False,
    sizes=(100, 1000),   # Rozpiętość wielkości kropek
    palette='viridis',
    alpha=0.7
)

# Dodajemy podpisy do każdej kropki (żeby wiedzieć, co jest czym)
for i in range(len(df_plot)):
    plt.text(
        df_plot.streams.iloc[i] + 1,
        df_plot.hours.iloc[i] + 0.1,
        df_plot.song.iloc[i],
        fontsize=9
    )

plt.title('Relacja: Liczba odtworzeń vs Czas słuchania', fontsize=16)
plt.xlabel('Liczba odtworzeń (x)', fontsize=12)
plt.ylabel('Suma godzin (h)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:

def wrap_labels(labels,  max_width):
    return[ textwrap.fill(label, max_width) for label in labels ]


top_album = ( df.groupby(['artist', 'album'])['ms_played'].
              sum().
              sort_values(ascending= False ).
              reset_index())
top_album['hours'] = (top_album['ms_played'] / 3600000) .round(1)

albums_plot = top_album.head(15).sort_values('hours')
wrapped_labels = wrap_labels(albums_plot['album'], 20)

print(top_album)
plt.figure(figsize=(12, 10))
plt.hlines( y = wrapped_labels, xmin= 0 , xmax = albums_plot['hours'], color='purple', alpha=1)
plt.scatter( x= albums_plot['hours'], y = wrapped_labels, color='green', s= 100, alpha=1)
for i, (h,a) in enumerate (zip(albums_plot['hours'], albums_plot['artist'])):
    plt.text(h+0.5, i , f"{a}", )
plt.xlabel('suma godzin')
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()


In [ ]:

print( df['reason_start'].unique())
print(df['reason_end'].unique())
reasons = get_top_ranking(df, 'reason_start', 10 )
reasons_end = get_top_ranking(df, 'reason_end', 10 )
print(reasons_end)

In [ ]:
skips = df[df['reason_end']== 'fwdbtn']
most_skipped=( skips.groupby(['artist','song']).size().sort_values(ascending=False))

clicked = df[df['reason_start']== 'clickrow']
most_clicked = ( clicked.groupby(['artist','song']).size().sort_values(ascending=False))



In [64]:
df_tmp = df.copy()

df_tmp['if_skip'] = np.where(
    (df_tmp['reason_end'] == 'fwdbtn') & (df_tmp['ms_played'] < 30000),
    1,
    0
)
skip_analysis = df_tmp.groupby(['artist','song']).agg(
                skip_count = ('if_skip', 'sum'),
                total_plays = ('ms_played', 'count')
).reset_index()

skip_analysis['skips%rate'] = ( skip_analysis['skip_count'] / skip_analysis['total_plays'] * 100 )

most_annoying = skip_analysis[skip_analysis['total_plays']>10].sort_values(by= 'skips%rate' , ascending=False).head(40)
display(most_annoying)

,artist,song,skip_count,total_plays,skips%rate
16779,Troye Sivan,Postcard (feat. Gordi),10,13,76.923077
3908,Dove Cameron,Ways to Be Wicked,8,12,66.666667
2602,Cavetown,Lemon Boy,11,17,64.705882
4369,Elvis Presley,Almost Always True,7,11,63.636364
4393,Elvis Presley,Slicin' Sand,7,11,63.636364
14549,Stanisław Moniuszko,Dziad i baba,9,15,60.000000
16421,Tom Odell,Wrong Crowd,13,22,59.090909
3375,Daniel Merriweather,Water And A Flame (feat. Adele),7,12,58.333333
14921,TACONAFIDE,Ekodiesel - Remix,7,12,58.333333
15704,The Honeysticks,Out Like a Light,11,20,55.000000
